In [ ]:
!hf auth login --token "REMOVED"

In [ ]:
import numpy as np
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [ ]:
PROMPT_TOKEN_LENGTH = 50
TOKEN_LIMIT = 128
GAMMA = 3

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
target_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B")
draft_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test").shuffle(seed=42)

# filter out empty lines and section headers
prompts = [x["text"][:] for x in ds if len(x["text"]) > 200 and not x["text"].startswith(" =")][:20]

In [ ]:
def tokenize_and_trim_prompt(prompt):
    encoded_prompt = tokenizer(prompt)
    return {
        'input_ids': torch.tensor([encoded_prompt.input_ids[:PROMPT_TOKEN_LENGTH]]), 
        'attention_mask': torch.tensor([encoded_prompt.attention_mask[:PROMPT_TOKEN_LENGTH]])
        }

tokenized_prompts = [tokenize_and_trim_prompt(prompt) for prompt in prompts]

tokenized_prompts[0]

In [ ]:
def sample_and_get_probas(logits, temperature=1.0):
    if temperature == 0.0: # Greedy decoding
        sample_id = logits.argmax(dim=-1)

        probas = torch.zeros_like(logits)
        probas.scatter_(
            dim=-1,
            index=sample_id.unsqueeze(-1),
            value=1.0
        )
        return sample_id, probas
    else:
        probas = torch.softmax(logits / temperature, dim=-1)
        sample_id = torch.multinomial(probas, num_samples=1)
        return sample_id.squeeze(), probas

def autoregressive_decoding_no_kv(model, input_ids, temperature=1.0, device='cuda', token_limit=TOKEN_LIMIT):
    prompt_length = input_ids.shape[0]
    total_length = prompt_length + token_limit

    sequence = torch.zeros(total_length, dtype=torch.long, device=device)
    sequence[:prompt_length] = input_ids
    
    current_length = prompt_length
    draft_probas_list = [] # Accumulate probabilities step-by-step
    
    model.to(device)
    with torch.no_grad():
        while current_length < total_length:
            current_input = sequence[:current_length].unsqueeze(0)
            
            output = model(current_input)
            
            last_token_logits = output.logits[0, -1, :]
            next_token_id, probas = sample_and_get_probas(last_token_logits, temperature)
            
            sequence[current_length] = next_token_id
            draft_probas_list.append(probas)
            current_length += 1
            
    return sequence[prompt_length:], torch.stack(draft_probas_list)

def autoregressive_decoding(model, input_ids, temperature=1.0, device='cuda', token_limit=TOKEN_LIMIT):
    prompt_length = input_ids.shape[0]
    total_length = prompt_length + token_limit

    sequence = torch.zeros(total_length, dtype=torch.long, device=device)
    sequence[:prompt_length] = input_ids
    
    current_length = prompt_length
    draft_probas_list = [] # Accumulate probabilities step-by-step
    
    model.to(device)
    with torch.no_grad():
        output = model(sequence[:prompt_length].unsqueeze(0), use_cache=True)
        past_key_values = output.past_key_values

        last_token_logits = output.logits[0, -1, :]
        next_token_id, probas = sample_and_get_probas(last_token_logits, temperature)

        sequence[current_length] = next_token_id
        draft_probas_list.append(probas)
        current_length += 1
        
        while current_length < total_length:
            current_input = sequence[current_length-1].unsqueeze(0).unsqueeze(0)
            
            output = model(current_input, past_key_values=past_key_values, use_cache=True)
            past_key_values = output.past_key_values
            
            last_token_logits = output.logits[0, -1, :]
            next_token_id, probas = sample_and_get_probas(last_token_logits, temperature)
            
            sequence[current_length] = next_token_id
            draft_probas_list.append(probas)
            current_length += 1
            
    return sequence[prompt_length:], torch.stack(draft_probas_list)

def truncate_kv_cache(past_key_values, n_keep): 
    if hasattr(past_key_values, 'crop'): # Modern approach: use the built-in crop method if available (transformers >= 4.36)
        past_key_values.crop(n_keep)
        return past_key_values
    else: # Legacy fallback: standard tuple approach (transformers < 4.36)
        new_past = []                                      
        for layer in past_key_values:
            k = layer[0][:, :, :n_keep, :]
            v = layer[1][:, :, :n_keep, :]
            new_past.append((k, v))
        return tuple(new_past)

def speculative_decoding(draft_model, target_model, encoded_prompt, temperature=1.0, device='cuda'):
    input_ids = encoded_prompt['input_ids'].squeeze(0)
    
    prompt_length = input_ids.shape[0]
    total_length = prompt_length + TOKEN_LIMIT

    sequence = torch.zeros(total_length + GAMMA, dtype=torch.long, device=device)
    sequence[:prompt_length] = input_ids

    current_length = prompt_length
    
    draft_model.to(device), target_model.to(device)
    with torch.no_grad():
        draft_out = draft_model(sequence[:prompt_length].unsqueeze(0), use_cache=True)
        draft_past = draft_out.past_key_values
    
        target_out = target_model(sequence[:prompt_length].unsqueeze(0), use_cache=True)
        target_past = target_out.past_key_values

        draft_past = truncate_kv_cache(draft_past, prompt_length - 1) # this is a neat trick 
        target_past = truncate_kv_cache(target_past, prompt_length - 1) 
        
        while current_length < total_length:
            draft_ids, draft_probas = [], []

            for i in range(GAMMA):
                current_input = sequence[current_length + i - 1].unsqueeze(0).unsqueeze(0)
                
                output = draft_model(current_input, past_key_values=draft_past, use_cache=True)
                draft_past = output.past_key_values
                
                last_token_logits = output.logits[0, -1, :]
                next_token_id, probas = sample_and_get_probas(last_token_logits, temperature)

                sequence[current_length + i] = next_token_id
                draft_ids.append(next_token_id)
                draft_probas.append(probas)
            draft_ids, draft_probas = torch.stack(draft_ids), torch.stack(draft_probas)
            # draft_ids, draft_probas = autoregressive_decoding(draft_model, sequence[:current_length], temperature, token_limit=GAMMA)
            
            sequence[current_length:current_length+GAMMA] = draft_ids
            target_last = sequence[current_length-1:current_length+GAMMA].unsqueeze(0)
            
            output = target_model(target_last, past_key_values=target_past, use_cache=True)
            target_past = output.past_key_values
            
            logits = output.logits[0, -(GAMMA + 1):, :]
            target_ids, target_probas = sample_and_get_probas(logits, temperature)

            n = GAMMA
            for idx in range(GAMMA):
                draft_token = draft_ids[idx]
                p_i = target_probas[idx, draft_token]
                q_i = draft_probas[idx, draft_token]

                r_i = torch.rand(1, device=device)[0] 
                
                if r_i > (p_i / q_i):
                    n = idx
                    break
            
            adjusted_distribution = target_probas[n, :]
            if n < GAMMA:
                p_probas = target_probas[n, :]
                q_probas = draft_probas[n, :]
                adjusted_distribution = torch.max(torch.zeros_like(p_probas), p_probas - q_probas)
                adjusted_distribution = adjusted_distribution / torch.sum(adjusted_distribution)
            final_id = torch.multinomial(adjusted_distribution, num_samples=1).squeeze()
            
            sequence[current_length + n] = final_id # this works as draft_ids[:n] == original_ids[:n] and they already sit in the sequence buffer
            
            valid_length = current_length + n + 1
            if n == GAMMA:
                last_draft_input = sequence[current_length + GAMMA - 1].unsqueeze(0).unsqueeze(0)
                with torch.no_grad():
                    draft_out = draft_model(last_draft_input, past_key_values=draft_past, use_cache=True)
                    draft_past = draft_out.past_key_values
                    
            draft_past = truncate_kv_cache(draft_past, valid_length - 1)
            target_past = truncate_kv_cache(target_past, valid_length - 1)
            
            current_length = valid_length
            
    return sequence[:total_length]
            
speculative_ids = speculative_decoding(draft_model, target_model, tokenized_prompts[0], temperature=0.0)     
autoregressive_ids, _ = autoregressive_decoding_no_kv(target_model, tokenized_prompts[0]['input_ids'].squeeze(0), temperature=0.0)

In [ ]:
(speculative_ids[50:] == autoregressive_ids).sum() == TOKEN_LIMIT